# 05 — Tempo de chegada e avaliação de incerteza
> **Estudo educacional independente; não substitui o PAE oficial, a Defesa Civil ou laudo de engenharia.**

Tempos: PAE p. 38 (quadro) e p. 55 (Figura 12). Definição: instante em que a onda atinge sobrelevação de 0,61 m.

In [1]:
import sys, warnings; warnings.filterwarnings('ignore')
sys.path.insert(0, '../src')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
from jurumirim import DISCLAIMER
print(DISCLAIMER)

Estudo educacional independente; não substitui o PAE oficial, a Defesa Civil ou laudo de engenharia.


In [2]:
from jurumirim import data, plots
sec = data.sections()
sec['celeridade_media_ms'] = sec.distancia_km.diff()*1000/(sec.tempo_chegada_min.diff()*60)
sec[['id','distancia_km','tempo_chegada_hhmm','celeridade_media_ms']].round(2)

,id,distancia_km,tempo_chegada_hhmm,celeridade_media_ms
0,ST-001,0.00,00:01,NaN
1,ST-002,0.82,00:03,6.83
2,ST-003,1.87,00:09,2.92
3,ST-004,3.34,00:13,6.12
4,ST-005,5.44,00:21,4.38
5,ST-006,6.71,00:31,2.12
6,ST-007,9.78,00:37,8.53
7,ST-008,11.41,00:45,3.40
8,ST-009,13.69,00:56,3.45
9,ST-010,16.42,01:09,3.50


A 'celeridade média' entre seções (distância ao longo do rio ÷ intervalo de tempo) é **derivada** dos valores do PAE, só para leitura; **não** é um resultado hidráulico novo.

In [3]:
from IPython.display import display
display(plots.section_charts())

<Figure size 1100x750 with 4 Axes>

## Incerteza posicional da mancha
Deslocamos a envoltória (buffer ±d) para ver se a conclusão espacial sobre a mancha urbanizada de Piraju mudaria.

In [4]:
from jurumirim import data
env = data.envelope(); mun = data.municipios(); urb = data.urbanizadas()
from jurumirim import gis
pir_m = mun[mun.codarea == gis.PIRAJU_CODE].geometry.iloc[0]
ids = data.urban_summary()
u = urb[urb.representative_point().within(pir_m)].copy()
main = u.loc[u.area.idxmax()].geometry
rest = u.drop(u.area.idxmax()).geometry.union_all()
rows = []
for d in (0, 10, 25, 50, 100, 250, 500, 1000):
    e = env.buffer(d)
    rows.append({'buffer_m': d, 'area_envoltoria_km2': e.area/1e6, 'maior_poligono_urbano_inter_km2': e.intersection(main).area/1e6, 'demais_poligonos_inter_km2': e.intersection(rest).area/1e6})
pd.DataFrame(rows).round(3)

,buffer_m,area_envoltoria_km2,maior_poligono_urbano_inter_km2,demais_poligonos_inter_km2
0,0,30.390,0.0,0.003
1,10,31.808,0.0,0.007
2,25,33.920,0.0,0.018
3,50,37.378,0.0,0.047
4,100,44.100,0.0,0.116
5,250,63.201,0.0,0.308
6,500,90.439,0.0,0.480
7,1000,133.206,0.0,0.951


Incertezas somadas: registro da grade (~0,5 m), vetorização (1-3 px ≈ 3,5-10 m), simplificação (2,5 m), incerteza do próprio PAE (modelo de terreno e escala 1:30.000 → ~15-30 m de largura de traço). A distância de ~2 km até o maior polígono urbanizado é **~2 ordens de grandeza maior** que a incerteza posicional (a conclusão 'sem interseção' para ele é robusta até ~1 km de buffer), enquanto as sobreposições marginais dos polígonos periféricos pequenos **não são robustas** (crescem com o buffer). Tudo vale **apenas para o cenário do PAE**.

In [5]:
print('Distância do maior polígono urbanizado à envoltória (m):', round(main.distance(env)))

Distância do maior polígono urbanizado à envoltória (m): 1962
